# 03 - Exploratory data analysis (EDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/03_eda.ipynb)

**Goal:** understand patterns in demand: weekly and yearly cycles, events, price, weather, how concentrated sales are.

Each chart has a short note on what to look for. At the end the notebook writes a report to `reports/`.

How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

In [ ]:
DATASET = "synthetic"    # "synthetic" | "synthetic_calm" | "m5" | "m5_christmas" | "walmart" | "walmart_synthetic"
from demand_agents.config import load_named
cfg = load_named(DATASET)
print("dataset:", DATASET, "| source:", cfg["data"]["source"], "| frequency:", cfg["data"]["frequency"])

In [ ]:
from demand_agents.data.loaders import load_dataset
from demand_agents.data.audit import audit_panel
from demand_agents.eda.tables import run_eda
from demand_agents.eda.plots import make_figures
df = load_dataset(cfg)                       # reads data/processed/ if you ran notebook 02, otherwise builds in memory
tables = run_eda(df)
figs = make_figures(tables, df, "reports/figures", DATASET)
print(f"{len(df):,} rows, {df['series_id'].nunique()} series, {df['date'].min().date()} to {df['date'].max().date()}")

## Total demand over time
**Look for:** trend, seasonal waves, sudden jumps or drops (shocks).

In [ ]:
display(Image(figs["total_demand"]))

## Weekday and month patterns
**Look for:** bars far from 1.0. Big weekday differences mean the model must know the day of the week; a month pattern means yearly seasonality.

In [ ]:
display(Image(figs["seasonality"]))

## Events (holidays)
**Look for:** how much higher or lower sales are on event days.

In [ ]:
display(Image(figs["events"])) if "events" in figs else print("no event information in this dataset")

## Demand types
Each dot is one item. **Right of the vertical line** = many zero days (intermittent); **above the horizontal line** = very variable order sizes. Dots at the bottom left are the easy, steady items.

In [ ]:
display(Image(figs["intermittency"]))

## Price effect
**Look for:** sales falling as price rises (bars falling from left to right means people buy more when it is cheaper).

In [ ]:
display(Image(figs["price_response"])) if "price_response" in figs else print("no price variation in this dataset")

## Weather effect
**Look for:** a slope. A flat line means temperature barely moves sales for these items.

In [ ]:
display(Image(figs["weather_response"])) if "weather_response" in figs else print("no weather in this table yet (merge NOAA in notebooks 01-02)")

## Concentration, zeros and memory

In [ ]:
for k in ("volume_concentration", "zero_share", "autocorrelation"):
    display(Image(figs[k]))
display(tables["abc_summary"])

ABC classes: **A** items make the first 80% of volume, **B** the next 15%, **C** the last 5%. Autocorrelation shows how much today's total demand resembles the demand 1, 7, 14, 28 days ago.

## What this means for modelling

In [ ]:
from demand_agents.eda.report import modelling_implications, write_report
audit = audit_panel(df, cfg["data"]["frequency"], DATASET, cfg["data"]["season_period"])
display(Markdown("\n".join("- " + s for s in modelling_implications(audit, tables))))

## Save the report

In [ ]:
import json, pathlib
log_p = pathlib.Path(f"reports/tables/cleaning_log_{DATASET}.csv")
js = pathlib.Path(f"reports/tables/data_audit_{DATASET}.json")
cleaning = pd.read_csv(log_p).fillna("").to_dict("records") if log_p.exists() else None
wa = json.loads(js.read_text()).get("weather_audit") if js.exists() else None
write_report(f"reports/data_audit_{DATASET}.md", audit, tables, cleaning, wa, figs)
print(f"saved reports/data_audit_{DATASET}.md and the figures in reports/figures/")

**Next:** `04_integration_and_features.ipynb`.